# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library, leveraging entity `@id` fields throughout.

### Dataset Source
The dataset is structured according to the Croissant schema specification, accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. We will use the Croissant schema URL and examine basic information about the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and manifest
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available *record sets* and their schema. Each record set and its fields are uniquely identified by an `@id`, which you should use in further exploration and extraction.

In [ ]:
# List all record sets in the dataset along with their @id and fields
record_sets = []

print("Available record sets and fields in the dataset (by @id):\n")
for record_set in dataset.record_sets:
    print(f"- Record Set @id: {record_set['@id']}")
    record_sets.append(record_set['@id'])
    if 'field' in record_set:
        fields = record_set['field']
        # If only one field, wrap in list
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields (@id):")
        for field in fields:
            print(f"    • {field['@id']}")
    print("")
# For downstream Notebook variables:
record_sets_ids = record_sets

## 3. Data Extraction
Let's load all records from each available record set (`record_sets_ids`) into pandas DataFrames. Each DataFrame is keyed by the respective record set's `@id`.

**Note:** You can use the printed `@id` values from the overview above to select specific record sets and fields for targeted exploration.

In [ ]:
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows for Record Set @id: {record_set_id}")
    print(f"Columns (by @id): {df.columns.tolist()}")
    print("")

# For demonstration, pick the first record set for further analysis
selected_record_set_id = record_sets_ids[0] if record_sets_ids else None
# Display a few records from the selected record set, if available
if selected_record_set_id and not dataframes[selected_record_set_id].empty:
    print(f"Sample records from Record Set: {selected_record_set_id}")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process the main record set. We'll select a numeric field (by `@id`) for basic filtering and normalization.

**Instructions:**
1. Identify which columns are numeric using pandas' dtypes (see code output).
2. Use the `@id` of your desired numeric column for all operations.

In [ ]:
# Automatically try to pick the first numeric column by dtype, fallback to manual selection if needed.
df = dataframes[selected_record_set_id]

# Find candidate numeric fields by dtype
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback: try to parse columns with digits as floats
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        except Exception:
            continue

if numeric_field_id is not None:
    print(f"Numeric field selected for EDA: {numeric_field_id}")
    threshold = df[numeric_field_id].mean()  # Use mean as a threshold demo
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt grouping by another field (pick a non-numeric column by @id)
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
            group_field_id = col
            break

    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric fields found in this record set!")

## 5. Visualization
Let's visualize the distribution of the selected numeric field. If grouping is available, plot means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.xticks(rotation=30)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load and parse a Croissant-structured dataset using `mlcroissant` and entity `@id` references
- Discover available record sets and fields (`@id`)
- Extract, filter, and normalize data by `@id`
- Perform basic grouping and exploratory analysis
- Visualize the distribution of numeric data fields

This approach ensures transparent and systematic handling of FAIR-compliant research datasets. You can customize the workflow based on the specific semantics and research questions associated with each record set and field.

**Next steps:**
- Explore additional record sets by specifying their `@id`
- Join among record sets if dataset relationships exist
- Perform further statistical or machine learning analyses